# Optional: CPU and WebGPU transcription measurements

Use this notebook to check the compiled encoder, decoder logits, transcript and
complete token sequence against eager Transformers/PyTorch. The main demos do
not run these comparisons.

Run once with `DEVICE = 'cpu'`, then with `DEVICE = 'webgpu'` in a fresh kernel.
Run one notebook at a time. Saved outputs come from a local browser run;
running it again measures your own device.


In [2]:
import json
import statistics
from IPython.display import Markdown, display
import time
import wave
from pathlib import Path

import numpy as np
import torch
from transformers import WhisperFeatureExtractor, WhisperForConditionalGeneration, WhisperTokenizer
from whisper_graphs import AudioEncoder, DecoderStep, empty_self_cache
from speech_pipeline import WhisperSpeech

DEVICE = 'webgpu'
MODEL_DIR = Path('/models/whisper-base')
torch.autograd.set_multithreading_enabled(False)
print(f'PyTorch {torch.__version__} | {DEVICE}')


/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/lib/python3.14/site-packages/huggingface_hub/utils/_fixes.py:22: UserWarning: only soft file lock is available
  from filelock import BaseFileLock, FileLock, SoftFileLock, Timeout


PyTorch 2.15.0a0 | webgpu


## 1. Load the original model and sample

Model loading includes the verified download or cache read. Tokenizer files and the reviewed Transformers bundle are preloaded.


In [3]:
from whisper_weights import ensure_checkpoint

model = None
load_started = time.perf_counter()
await ensure_checkpoint(MODEL_DIR)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_DIR, local_files_only=True, use_safetensors=False,
    attn_implementation='eager').eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
extractor = WhisperFeatureExtractor.from_pretrained(MODEL_DIR, local_files_only=True)

weights_and_model_load_seconds = time.perf_counter() - load_started

with wave.open('jfk.wav', 'rb') as source:
    assert source.getframerate() == 16000 and source.getnchannels() == 1
    sample_audio = np.frombuffer(source.readframes(source.getnframes()), dtype='<i2').astype(np.float32) / 32768
features = extractor(sample_audio, sampling_rate=16000,
                     return_tensors='pt', return_attention_mask=True)
print(f'{sum(p.numel() for p in model.parameters()):,} parameters')
print(f'{len(sample_audio) / 16000:.1f} seconds of audio → {tuple(features.input_features.shape)} features')


Loading Whisper base weights (first download: 277 MiB)...
Using verified cached weights.
72,593,920 parameters
11.0 seconds of audio → (1, 80, 3000) features


## 2. Check against eager PyTorch

The reference runs on CPU. We compare encoder output, first decoder logits and
the complete greedy token sequence. Both devices use the same original FP32
weights and checkpoint suppression rules.


In [5]:
reference_started = time.perf_counter()
prefix = [model.generation_config.decoder_start_token_id] + [token for _, token in
    tokenizer.get_decoder_prompt_ids(language='en', task='transcribe', no_timestamps=True)]
with torch.inference_mode():
    expected_generated = model.generate(
        features.input_features, attention_mask=features.attention_mask,
        language='en', task='transcribe', return_timestamps=False,
        do_sample=False, max_new_tokens=124)
    expected_hidden, expected_cross = AudioEncoder(model)(features.input_features)
    expected_logits, _ = DecoderStep(model, 128)(
        torch.tensor([prefix]), torch.arange(len(prefix)), expected_hidden,
        empty_self_cache(model.config, 128, 'cpu'), expected_cross)
expected_text = tokenizer.batch_decode(expected_generated, skip_special_tokens=True)[0].strip()
expected_ids = expected_generated[0].tolist()
if expected_ids[-1] == model.generation_config.eos_token_id:
    expected_ids = expected_ids[:-1]
if expected_ids[:len(prefix)] == prefix:
    expected_ids = expected_ids[len(prefix):]
del expected_cross
print(expected_text)
eager_reference_seconds = time.perf_counter() - reference_started
print(f'Eager CPU reference checks: {eager_reference_seconds:.2f} s')

device_setup_started = time.perf_counter()
if DEVICE == 'webgpu':
    from iree.runtime.webgpu import get_webgpu_device
    from iree.turbine.runtime.torch_device import TorchDevice
    gpu = TorchDevice(await get_webgpu_device(
        max_concurrent_operations=len(model.state_dict()) + 32), name='webgpu')
else:
    gpu = None
model = model.to(DEVICE)
if gpu is not None:
    await gpu.synchronize()
speech = WhisperSpeech(model, tokenizer, extractor, device=DEVICE, gpu=gpu)

device_setup_seconds = time.perf_counter() - device_setup_started
print('Compiling and transcribing the sample...', flush=True)
started = time.perf_counter()
cold = await speech.transcribe(sample_audio)
cold_seconds = time.perf_counter() - started
assert cold['token_ids'] == expected_ids and cold['text'] == expected_text
assert not cold['cancelled'] and not cold['truncated']

checked = await speech.transcribe(sample_audio, retain_reference=True)
torch.testing.assert_close(checked['encoder_reference'], expected_hidden, rtol=3e-3, atol=1e-3)
torch.testing.assert_close(checked['first_logits'], expected_logits, rtol=3e-3, atol=3e-3)
assert checked['token_ids'] == expected_ids and checked['text'] == expected_text
reference_errors = dict(encoder=float((checked['encoder_reference'] - expected_hidden).abs().max()),
                        first_logits=float((checked['first_logits'] - expected_logits).abs().max()))
del checked
print('Compiled encoder, decoder logits and complete token sequence match eager PyTorch.')


And so my fellow Americans, ask not what your country can do for you, ask what you can do for your country.
Eager CPU reference checks: 7.33 s
Compiling and transcribing the sample...


/lib/python3.14/site-packages/torch/fx/interpreter.py:177: TqdmMonitorWarning: tqdm:disabling monitor support (monitor_interval = 0) due to:
can't start new thread
  pbar = tqdm(


Compiled encoder, decoder logits and complete token sequence match eager PyTorch.


## 3. Measure completed transcriptions

The first transcription includes compilation and normal execution. Additional
reference readbacks run separately from the timings. Each warmed run starts with
fresh self-attention and cross-attention caches.

Real-time factor is processing time divided by audio duration; lower is faster.
Recordings and initial package/model downloads are excluded from warmed timings.
Encoder and decoder timings include their completion waits; decoder timing also
includes logits readback and Python token selection.


In [6]:
measurements = []
for _ in range(3):
    completed = await speech.transcribe(sample_audio)
    assert completed['text'] == expected_text and completed['token_ids'] == expected_ids
    assert not completed['cancelled'] and not completed['truncated']
    measurements.append(completed)
warm = sorted(measurements, key=lambda result: result['transcription_seconds'])[1]
report = dict(passed=True, model='openai/whisper-base', device=DEVICE, precision='FP32',
              pytorch=torch.__version__, weights_and_model_load_seconds=weights_and_model_load_seconds,
              eager_cpu_reference_seconds=eager_reference_seconds,
              device_setup_seconds=device_setup_seconds,
              compile_and_first_transcription_seconds=cold_seconds,
              audio_seconds=warm['audio_seconds'], transcription_seconds=warm['transcription_seconds'],
              real_time_factor=warm['transcription_seconds'] / warm['audio_seconds'],
              transcription_samples_seconds=[result['transcription_seconds'] for result in measurements],
              timings=warm['timings'], transcript=warm['text'], token_ids=warm['token_ids'],
              max_abs_error=reference_errors)

display(Markdown(
    f"| {DEVICE.upper()} measurement | Time |\n| :--- | ---: |\n"
    f"| Download/cache and load model | {weights_and_model_load_seconds:.2f} s |\n"
    f"| Eager CPU reference checks | {eager_reference_seconds:.2f} s |\n"
    f"| Device setup and initial upload | {device_setup_seconds:.2f} s |\n"
    f"| Compile and first transcription | {cold_seconds:.2f} s |\n"
    f"| Warmed transcription, fresh caches (median of 3) | {warm['transcription_seconds']:.2f} s |\n"
    f"| Audio duration | {warm['audio_seconds']:.2f} s |\n"
    f"| Real-time factor (lower is faster) | {report['real_time_factor']:.2f} |"
))
print(json.dumps(report, indent=2))
_ = Path(f'benchmark-{DEVICE}.json').write_text(json.dumps(report, indent=2) + "\n")


| WEBGPU measurement | Time |
| :--- | ---: |
| Download/cache and load model | 0.66 s |
| Eager CPU reference checks | 7.33 s |
| Device setup and initial upload | 4.45 s |
| Compile and first transcription | 17.06 s |
| Warmed transcription, fresh caches (median of 3) | 0.65 s |
| Audio duration | 11.00 s |
| Real-time factor (lower is faster) | 0.06 |

{
  "passed": true,
  "model": "openai/whisper-base",
  "device": "webgpu",
  "precision": "FP32",
  "pytorch": "2.15.0a0",
  "weights_and_model_load_seconds": 0.6623999999999999,
  "eager_cpu_reference_seconds": 7.331700000000001,
  "device_setup_seconds": 4.4541,
  "compile_and_first_transcription_seconds": 17.064899999999998,
  "audio_seconds": 11.0,
  "transcription_seconds": 0.6546999999999983,
  "real_time_factor": 0.059518181818181665,
  "transcription_samples_seconds": [
    0.6640000000000015,
    0.6546999999999983,
    0.6135000000000019
  ],
  "timings": {
    "features_ms": 16.59999999999684,
    "encoder_ms": 230.80000000000211,
    "decoder_ms": 400.0999999999948
  },
  "transcript": "And so my fellow Americans, ask not what your country can do for you, ask what you can do for your country.",
  "token_ids": [
    400,
    370,
    452,
    7177,
    6280,
    11,
    1029,
    406,
    437,
    428,
    1941,
    393,
    360,
    337,
    291,
    11,
    1029,
    437,

## Recorded CPU and WebGPU comparison

Both runs use the same audio, FP32 model and greedy decoding. Performance on your
own device may differ. The saved report includes machine and browser details.


In [9]:
record = json.loads(Path('benchmark-results.json').read_text())
cpu = record['compiled_transcription']['cpu']
webgpu = record['compiled_transcription']['webgpu']
display(Markdown(
    "| Measurement | CPU | WebGPU |\n| :--- | ---: | ---: |\n"
    f"| Compile and first transcription | {cpu['compile_and_first_transcription_seconds']:.2f} s | "
    f"{webgpu['compile_and_first_transcription_seconds']:.2f} s |\n"
    f"| Warmed transcription (median of 3) | {cpu['transcription_seconds']:.2f} s | "
    f"{webgpu['transcription_seconds']:.2f} s |\n"
    f"| Real-time factor (lower is faster) | {cpu['real_time_factor']:.2f} | "
    f"{webgpu['real_time_factor']:.2f} |\n"
    f"| Audio features | {cpu['timings']['features_ms']:.1f} ms | {webgpu['timings']['features_ms']:.1f} ms |\n"
    f"| Encoder | {cpu['timings']['encoder_ms']:.1f} ms | {webgpu['timings']['encoder_ms']:.1f} ms |\n"
    f"| Decoder and token selection | {cpu['timings']['decoder_ms']:.1f} ms | {webgpu['timings']['decoder_ms']:.1f} ms |"
))


| Measurement | CPU | WebGPU |
| :--- | ---: | ---: |
| Compile and first transcription | 22.16 s | 17.06 s |
| Warmed transcription (median of 3) | 5.79 s | 0.65 s |
| Real-time factor (lower is faster) | 0.53 | 0.06 |
| Audio features | 10.6 ms | 16.6 ms |
| Encoder | 4753.3 ms | 230.8 ms |
| Decoder and token selection | 1019.0 ms | 400.1 ms |